# BTK Datathon 2026 — Data Understanding & Baseline Notebook

This notebook is designed as the **first clean notebook** for the project.

Main goal:

1. Understand what the dataset contains
2. Check target distribution and missing values
3. Separate numeric, categorical, and text features
4. Build a safe baseline model
5. Create a valid submission file

This is not the final advanced model notebook.  
The purpose is to make the project explainable, reproducible, and safe before moving into heavy modeling.

## 1. Imports and Configuration

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import os
import numpy as np
import pandas as pd

from sklearn.model_selection import KFold, cross_val_score
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import mean_squared_error
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor

RANDOM_STATE = 42
pd.set_option("display.max_columns", 200)
pd.set_option("display.max_colwidth", 200)

## 2. Load Data

Expected local structure:

```text
data/
├── train.csv
├── test_x.csv
└── sample_submission.csv
```

In [ ]:
DATA_DIR = "../data"   # If your notebook is inside notebooks/
TRAIN_PATH = os.path.join(DATA_DIR, "train.csv")
TEST_PATH = os.path.join(DATA_DIR, "test_x.csv")
SAMPLE_PATH = os.path.join(DATA_DIR, "sample_submission.csv")

train = pd.read_csv(TRAIN_PATH)
test = pd.read_csv(TEST_PATH)
sample_submission = pd.read_csv(SAMPLE_PATH)

print("Train shape:", train.shape)
print("Test shape:", test.shape)
print("Sample submission shape:", sample_submission.shape)

display(train.head())
display(test.head())
display(sample_submission.head())

## 3. Basic Dataset Checks

In [ ]:
print("Train columns:")
print(train.columns.tolist())

print("\nTest columns:")
print(test.columns.tolist())

target_col = "career_success_score"
id_col = "student_id"
text_col = "mentor_feedback_text"

print("\nTarget exists in train:", target_col in train.columns)
print("Target exists in test:", target_col in test.columns)

print("\nDuplicated student IDs in train:", train[id_col].duplicated().sum())
print("Duplicated student IDs in test:", test[id_col].duplicated().sum())

print("\nTrain dtypes:")
display(train.dtypes.value_counts())

## 4. Target Analysis

The target is a continuous score between 0 and 100.  
Since the competition metric is MSE, large mistakes are punished heavily.

In [ ]:
display(train[target_col].describe())

print("Target min:", train[target_col].min())
print("Target max:", train[target_col].max())

train[target_col].hist(bins=40)

## 5. Missing Value Analysis

In [ ]:
missing_train = train.isna().sum().sort_values(ascending=False)
missing_train = missing_train[missing_train > 0]

missing_test = test.isna().sum().sort_values(ascending=False)
missing_test = missing_test[missing_test > 0]

print("Missing values in train:")
display(missing_train.to_frame("missing_count"))

print("Missing values in test:")
display(missing_test.to_frame("missing_count"))

## 6. Feature Groups

We separate the dataset into:

- ID column: not used for modeling
- Target column: prediction target
- Text column: handled with TF-IDF
- Numeric columns
- Categorical columns

In [ ]:
feature_cols = [c for c in train.columns if c not in [target_col]]
X = train[feature_cols].copy()
y = train[target_col].copy()
X_test = test.copy()

# Keep test IDs for submission
test_ids = X_test[id_col].copy()

# Drop ID from features
X = X.drop(columns=[id_col])
X_test_features = X_test.drop(columns=[id_col])

numeric_cols = X.drop(columns=[text_col]).select_dtypes(include=["int64", "float64"]).columns.tolist()
categorical_cols = X.drop(columns=[text_col]).select_dtypes(include=["object"]).columns.tolist()

print("Numeric columns:", len(numeric_cols))
print(numeric_cols)

print("\nCategorical columns:", len(categorical_cols))
print(categorical_cols)

print("\nText column:", text_col)

## 7. Quick Correlation Check for Numeric Features

This helps us understand which structured variables are most related to the target.

In [ ]:
corr_df = train[numeric_cols + [target_col]].corr(numeric_only=True)[target_col].sort_values(ascending=False)
display(corr_df.to_frame("correlation_with_target").head(20))
display(corr_df.to_frame("correlation_with_target").tail(20))

## 8. Text Feature Quick Look

The mentor feedback text should not be ignored because the competition statement explicitly highlights it.

In [ ]:
print("Example mentor feedback texts:")
display(train[[text_col, target_col]].sample(10, random_state=RANDOM_STATE))

## 9. Baseline 1 — Mean Prediction

This is the simplest possible baseline. Any real model should beat this.

In [ ]:
mean_pred = np.full(shape=len(y), fill_value=y.mean())
baseline_mse = mean_squared_error(y, mean_pred)
baseline_rmse = baseline_mse ** 0.5

print("Mean baseline MSE:", baseline_mse)
print("Mean baseline RMSE:", baseline_rmse)

## 10. Baseline 2 — Structured + Text Pipeline with Ridge

This baseline combines:

- Numeric features with median imputation and scaling
- Categorical features with most-frequent imputation and one-hot encoding
- Mentor feedback text with TF-IDF
- Ridge regression

Ridge is a good sanity-check model because it is stable and fast.

In [ ]:
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

text_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="constant", fill_value="")),
    ("tfidf", TfidfVectorizer(
        max_features=3000,
        ngram_range=(1, 2),
        min_df=2
    ))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_cols),
        ("cat", categorical_transformer, categorical_cols),
        ("text", text_transformer, text_col)
    ],
    remainder="drop"
)

ridge_model = Pipeline(steps=[
    ("preprocess", preprocessor),
    ("model", Ridge(alpha=10.0, random_state=RANDOM_STATE))
])

cv = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

neg_mse_scores = cross_val_score(
    ridge_model,
    X,
    y,
    scoring="neg_mean_squared_error",
    cv=cv,
    n_jobs=-1
)

mse_scores = -neg_mse_scores

print("Ridge + TF-IDF CV MSE scores:", mse_scores)
print("Mean CV MSE:", mse_scores.mean())
print("Mean CV RMSE:", np.sqrt(mse_scores.mean()))
print("Std CV MSE:", mse_scores.std())

## 11. Train Baseline Model on Full Training Data

In [ ]:
ridge_model.fit(X, y)
test_predictions = ridge_model.predict(X_test_features)

# The target is described as a 0-100 score, so we clip predictions safely.
test_predictions = np.clip(test_predictions, 0, 100)

print("Prediction min:", test_predictions.min())
print("Prediction max:", test_predictions.max())
print("Prediction mean:", test_predictions.mean())

## 12. Create Submission File

In [ ]:
submission = pd.DataFrame({
    id_col: test_ids,
    target_col: test_predictions
})

display(submission.head())
display(submission[target_col].describe())

os.makedirs("../submissions", exist_ok=True)
submission_path = "../submissions/submission_baseline_ridge_tfidf.csv"
submission.to_csv(submission_path, index=False)

print("Saved submission to:", submission_path)

## 13. Next Experiments

After this baseline works, stronger experiments can include:

1. CatBoost on structured features
2. LightGBM / XGBoost with engineered features
3. Better text features from mentor feedback
4. SentenceTransformer embeddings
5. Out-of-fold stacking
6. Weighted blending of stable models
7. Error analysis by department, target role, and university tier

Important: do not jump directly into complex models before this notebook runs from top to bottom.